In [ ]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

# ============================================================
# 0.Basic parameters
# ============================================================
data_file = r"./input/All_cohort_MAIN.csv"

duration_col = "time"
event_col = "status"
cohort_col = "cohort"
exposure_col = "BrainVital8"

base_covariates = [
    "sex", "age", "bmi",
    "drink",
    "hypertension"
]

# The data are actually only 7 cohort (NHANES is not in the file)
cohort_map = {
    "elsa": "ELSA",
    "charls": "charls",
    "hrs": "HRS",
    "klosa": "KLoSA",
    "mhas": "MHAS",
    "share": "share",
    "ukb": "ukb",
}

# ============================================================
# 1. Read data + cohort normalization (the only filtering occurs here)
# ============================================================
df = pd.read_csv(data_file, low_memory=False)
print("[INFO] Loaded:", df.shape)

df["_cohort_norm"] = (
    df[cohort_col].astype(str).str.strip().str.lower()
)

df = df[df["_cohort_norm"].isin(cohort_map.keys())].copy()
df[cohort_col] = df["_cohort_norm"].map(cohort_map)
df.drop(columns="_cohort_norm", inplace=True)

df[cohort_col] = df[cohort_col].astype("category")
print("\n[INFO] Cohort counts after normalization:")
print(df[cohort_col].value_counts())

# cluster/interaction copy
df["cluster_cohort"] = df[cohort_col]
df["cohort_for_interaction"] = df[cohort_col]

# ============================================================
# 2. Numerical variables: median imputation within cohort
# ============================================================
numeric_cols = [
    "age", "bmi", exposure_col,
    "sex", "smoke", "drink",
    "T2D", "hypertension", "depression",
    duration_col
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df.groupby(cohort_col, observed=False)[col].transform(
        lambda x: x.fillna(x.median())
    )

# ============================================================
# 3. income/education: Categorical variable (compatible with HRS Q1-Q4)
# ============================================================
for col in ["income", "education"]:
    df[col] = df[col].astype(str)
    df[col] = df.groupby(cohort_col, observed=False)[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )
    df[col] = df[col].fillna(df[col].mode()[0]).astype("category")

# ============================================================
# 4. Clean up the missing
# ============================================================
df = df.dropna(subset=[duration_col, event_col, exposure_col] + base_covariates).copy()
df[event_col] = df[event_col].astype(int)

# ============================================================
# 5. Zero-variance covariates were automatically removed
# ============================================================
def get_nonconstant_covariates(dff, covariates):
    keep, dropped = [], []
    for col in covariates:
        if dff[col].nunique(dropna=True) > 1:
            keep.append(col)
        else:
            dropped.append(col)
    return keep, dropped

covariates_keep, covariates_dropped = get_nonconstant_covariates(df, base_covariates)
print("\n[INFO] Dropped zero-variance covariates:", covariates_dropped)

# ============================================================
# 6. Main model (continuous BrainVital8)
# ============================================================
formula_main = exposure_col + " + " + " + ".join(covariates_keep)

cph_main = CoxPHFitter(penalizer=0.01)
cph_main.fit(
    df,
    duration_col=duration_col,
    event_col=event_col,
    strata=cohort_col,
    cluster_col="cluster_cohort",
    robust=True,
    formula=formula_main
)

print("\n=== Main mega-analysis model (continuous) ===")
cph_main.print_summary(decimals=3)
cph_main.summary.to_csv("./output/Cox_main_results.csv", encoding="utf-8-sig")

# ============================================================
# 7. Quartile model (quartile within cohort)
# ============================================================
df_q = df.copy()

def cohort_qcut(x):
    # When ties cause qcut error, downgrade processing
    try:
        return pd.qcut(x, q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
    except Exception:
        return pd.Series(["Q2"] * len(x), index=x.index)

df_q["BrainVital8_Q"] = (
    df_q.groupby(cohort_col, observed=False)[exposure_col]
        .transform(cohort_qcut)
        .astype("category")
)

df_q["BrainVital8_Q"] = df_q["BrainVital8_Q"].cat.set_categories(
    ["Q1", "Q2", "Q3", "Q4"], ordered=True
)

formula_quart = "BrainVital8_Q + " + " + ".join(covariates_keep)

cph_quart = CoxPHFitter(penalizer=0.01)
cph_quart.fit(
    df_q,
    duration_col=duration_col,
    event_col=event_col,
    strata=cohort_col,
    cluster_col="cluster_cohort",
    robust=True,
    formula=formula_quart
)

print("\n=== Quartile model (Q1 reference) ===")
cph_quart.print_summary(decimals=3)

quart_summary = (
    cph_quart.summary.loc[cph_quart.summary.index.str.contains("BrainVital8_Q")]
    [["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
)
quart_summary.index = ["Q2 vs Q1", "Q3 vs Q1", "Q4 vs Q1"]
quart_summary.to_csv("./output/Cox_quartile_results.csv", encoding="utf-8-sig")

# ============================================================
# 8. Interaction model (BrainVital8 × cohort)
# Note: When strata=cohort, formula cannot use cohort directly, so use cohort_for_interaction
# ============================================================
formula_interaction = (
    f"{exposure_col} * cohort_for_interaction + "
    + " + ".join([c for c in covariates_keep if c != "sex"])
)

cph_inter = CoxPHFitter(penalizer=0.01)
cph_inter.fit(
    df,
    duration_col=duration_col,
    event_col=event_col,
    strata=cohort_col,
    cluster_col="cluster_cohort",
    robust=True,
    formula=formula_interaction
)

print("\n=== Interaction model (BrainVital8 × cohort) ===")
cph_inter.print_summary(decimals=3)
cph_inter.summary.to_csv("./output/Cox_interaction_results.csv", encoding="utf-8-sig")

# ============================================================
# 9. LOCO mega-analysis (key fix: remove_unused_categories)
# ============================================================
loco_results = []

for c in df[cohort_col].cat.categories:
    print(f"\n--- LOCO: excluding cohort {c} ---")

    df_loco = df[df[cohort_col] != c].copy()

    # ✅ Key: Delete unused strata categories to avoid lifelines KeyError
    df_loco[cohort_col] = df_loco[cohort_col].cat.remove_unused_categories()
    df_loco["cluster_cohort"] = df_loco[cohort_col]
    df_loco["cohort_for_interaction"] = df_loco[cohort_col]

    cov_keep_loco, dropped_loco = get_nonconstant_covariates(df_loco, covariates_keep)
    print("[INFO] Dropped in LOCO due to zero variance:", dropped_loco)

    formula_loco = exposure_col + " + " + " + ".join(cov_keep_loco)

    cph_loco = CoxPHFitter(penalizer=0.01)
    cph_loco.fit(
        df_loco,
        duration_col=duration_col,
        event_col=event_col,
        strata=cohort_col,
        cluster_col="cluster_cohort",
        robust=True,
        formula=formula_loco
    )

    hr = cph_loco.summary.loc[exposure_col]
    loco_results.append({
        "excluded_cohort": c,
        "HR": hr["exp(coef)"],
        "lower_95": hr["exp(coef) lower 95%"],
        "upper_95": hr["exp(coef) upper 95%"],
        "p_value": hr["p"],
        "n": df_loco.shape[0],
        "events": int(df_loco[event_col].sum())
    })

df_loco_results = pd.DataFrame(loco_results)
df_loco_results.to_csv("./output/Cox_LOCO_results.csv", index=False, encoding="utf-8-sig")

print("\n=== LOCO mega-analysis completed ===")
print(df_loco_results)
print("\n✅ All analyses completed successfully.")
